## 📥 Twitter Sentiment Analysis
In this lab, you will build a tweet classification system using Natural Language Processing (NLP) techniques to classify Twitter tweets as either positive or negative.

🔍 What you will do:
- Load the dataset
- Split the data and vectorize the tweets
- Build and train a classification model
- Evaluate the model’s performance
- Test the model with new tweet examples

In [1]:
!pip install datasets scikit-learn textblob -q
!python -m textblob.download_corpora


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Finished.


[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\hayaa\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hayaa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hayaa\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\hayaa\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package conll2000 to
[nltk_data]     C:\Users\hayaa\AppData\Roaming\nltk_data...
[nltk_data]   Package conll2000 is already up-to-date!
[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\hayaa\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is alr

In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from textblob import TextBlob 

import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.porter import PorterStemmer

In [3]:
#import kagglehub

# Download latest version
#path = kagglehub.dataset_download("sahideseker/tweet-sentiment-classification-dataset")

#print("Path to dataset files:", path)

In [4]:
df = pd.read_csv('tweet_sentiment.csv') 

df.head()

,tweet,sentiment
0,The event starts at 5 PM.,neutral
1,I hate how this turned out.,negative
2,Fantastic experience!,positive
3,Fantastic experience!,positive
4,This is the worst thing ever!,negative


In [5]:
english_stopwords = stopwords.words('english')
stemmer = PorterStemmer()

# define cleaning function
def clean_review(text):
  # Tokenize the text and filter out non-alphabetic words
  words = [word for word in word_tokenize(text) if word.isalpha()]

  # Remove stopwords and stem the remaining words
  cleaned_words = [stemmer.stem(word) for word in words if word not in english_stopwords]

  # Join the cleaned words into a single string
  text = ' '.join(cleaned_words)

  return text

In [6]:
df['tweets_after_cleaning'] = df['tweet'].apply(clean_review)

In [7]:
text = df['tweets_after_cleaning'].values
label = df['sentiment'].values

X_train, X_test, y_train, y_test = train_test_split(text, label, train_size=0.5, test_size=0.5, random_state=42)
    
# vectorize to make the model understand the words in numrecal 
vectorizer = CountVectorizer(binary=True, max_features=10000) 


X_train_vec = vectorizer.fit_transform(X_train) # transform the trained data to vectore and fit the model in to it
X_test_vec = vectorizer.transform(X_test) # only transform the test data to vector. Note: no need to fit, we will test it

In [9]:
model = LogisticRegression()

# train the classifier on the training data
model.fit(X_train_vec,y_train)

# get the mean accuracy on the training data
acc_train = model.score(X_train_vec,y_train)

print('Training Accuracy:', acc_train)

Training Accuracy: 1.0


In [10]:
y_pred = model.predict(X_test_vec)
accuracy = model.score(X_test_vec, y_test)
print("✅ Accuracy: ", accuracy)


✅ Accuracy:  1.0


In [11]:
def correct_text(text):
    return str(TextBlob(text).correct())


def predict(model, vectorizer, review):
    review = correct_text(review)       
    review_bow = vectorizer.transform([review])
    sentiment = 'Positive 😊' if model.predict(review_bow)[0] == 'positive' else 'Negative 😞'
    return sentiment


In [12]:
review = 'i hate this'
predict(model, vectorizer, review)

'Negative 😞'